# Klasifikasi Penyakit Daun Paprika menggunakan Convolutional Neural Network (CNN)

Notebook ini digunakan untuk melatih dan mengevaluasi model deep learning CNN dalam mendeteksi dan mengklasifikasikan penyakit pada daun paprika. Eksperimen dibagi menjadi tiga tahap:
1. **Baseline Model** (Model dasar tanpa augmentasi)
2. **Model dengan Online Augmentation** (Augmentasi dinamis menggunakan layer Keras)
3. **Model dengan Offline Augmentation** (Augmentasi fisik menggunakan ImageDataGenerator)

## 1. Setup & Eksplorasi Data

Pada bagian ini, kita mengimpor seluruh library yang dibutuhkan serta memeriksa direktori dataset lokal.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report

# Konfigurasi Path Dataset
dataset_path = "Pepper_Dataset"
print("TensorFlow Version:", tf.__version__)
print("Dataset Path:", os.path.abspath(dataset_path))

# Tampilkan kelas dan jumlah gambar di masing-masing folder
if os.path.exists(dataset_path):
    for folder in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder)
        if os.path.isdir(folder_path):
            print(f"- {folder}: {len(os.listdir(folder_path))} gambar")
else:
    print("Error: Folder dataset tidak ditemukan!")

### Fungsi Helper untuk Prediksi Gambar

Kita mendefinisikan sebuah fungsi helper agar proses prediksi gambar pengujian lebih ringkas dan tidak berulang.

In [ ]:
def predict_and_plot(model_obj, img_path, class_names):
    if not os.path.exists(img_path):
        print(f"Error: Gambar {img_path} tidak ditemukan!")
        return
    
    # Load & preprocess
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    prediction = model_obj.predict(img_array)
    pred_idx = np.argmax(prediction)
    predicted_class = class_names[pred_idx]
    confidence = prediction[0][pred_idx] * 100
    
    # Plot
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Prediksi: {predicted_class} ({confidence:.2f}%)")
    plt.axis("off")
    plt.show()
    
    print(f"File: {img_path}")
    print(f"Hasil Prediksi: {predicted_class} ({confidence:.2f}%)")
    print(f"Probabilitas: {list(zip(class_names, prediction[0]))}\n")

## 2. Eksperimen 1: Baseline Model (Tanpa Augmentasi)

Melatih model CNN dasar tanpa menggunakan teknik augmentasi data untuk melihat performa awal model.

In [ ]:
# Load dataset training & validation
train_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Dapatkan nama kelas
class_names_inferred = train_data.class_names
print("Inferred Class Names:", class_names_inferred)

# Normalisasi pixel ke range [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
val_data = val_data.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Struktur Baseline Model
model_baseline = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_baseline.summary()

In [ ]:
model_baseline.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline = model_baseline.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

# Simpan model baseline
model_baseline.save("model_paprika.h5")

In [ ]:
# Plot accuracy & loss side-by-side
acc = history_baseline.history['accuracy']
val_acc = history_baseline.history['val_accuracy']
loss = history_baseline.history['loss']
val_loss = history_baseline.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

# Evaluasi menggunakan Confusion Matrix & Classification Report
y_true = []
y_pred = []
for images, labels in val_data:
    preds = model_baseline.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Standard class names dengan underscore untuk reporting
report_classes = ["Bacterial_spot", "Cercospora_leaf_spot", "Healthy"]
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=report_classes))

In [ ]:
# Uji model baseline pada paprika.jpg
predict_and_plot(model_baseline, "samples/paprika.jpg", report_classes)

## 3. Eksperimen 2: Model dengan Online Data Augmentation

Menambahkan layer augmentasi data bawaan Keras langsung di dalam struktur model untuk melakukan augmentasi gambar secara dinamis saat training.

In [ ]:
# Definisikan layer augmentasi dinamis
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2)
])

model_online_aug = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_online_aug.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_online_aug = model_online_aug.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

# Simpan model
model_online_aug.save("model_paprika_augmented.h5")

In [ ]:
# Plot history
acc = history_online_aug.history['accuracy']
val_acc = history_online_aug.history['val_accuracy']
loss = history_online_aug.history['loss']
val_loss = history_online_aug.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='train acc')
plt.plot(epochs_range, val_acc, label='val acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='train loss')
plt.plot(epochs_range, val_loss, label='val loss')
plt.legend()
plt.title('Loss')
plt.show()

# Evaluasi Metrics
y_true = []
y_pred = []
for images, labels in val_data:
    preds = model_online_aug.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=report_classes))

In [ ]:
# Uji model online augmentation
predict_and_plot(model_online_aug, "samples/paprika.jpg", report_classes)

## 4. Eksperimen 3: Offline Data Augmentation (ImageDataGenerator)

Melakukan augmentasi gambar secara offline menggunakan `ImageDataGenerator` untuk secara fisik menghasilkan dan menyimpan salinan gambar baru yang teraugmentasi langsung di dalam folder kelas `Cercospora Leaf Spot`.

In [ ]:
# Tentukan folder sumber augmentasi
source_folder = "Pepper_Dataset/Cercospora Leaf Spot"
print("Jumlah gambar Cercospora sebelum augmentasi:", len(os.listdir(source_folder)))

# Definisikan generator
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Proses augmentasi fisik (Simpan file baru ke folder)
current_images = [f for f in os.listdir(source_folder) if not f.startswith('aug')]
aug_images = [f for f in os.listdir(source_folder) if f.startswith('aug')]

if len(aug_images) == 0:
    print("Menjalankan offline data augmentation...")
    count = 0
    for filename in current_images:
        img_path = os.path.join(source_folder, filename)
        try:
            img = load_img(img_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)
            
            i = 0
            for batch in datagen.flow(
                x,
                batch_size=1,
                save_to_dir=source_folder,
                save_prefix='aug',
                save_format='jpg'
            ):
                i += 1
                count += 1
                if i >= 5:
                    break
        except Exception as e:
            print(f"Error pada {filename}: {e}")
    print("Total gambar hasil augmentasi baru:", count)
else:
    print("Augmentasi offline sudah pernah dijalankan sebelumnya (ditemukan file berawalan 'aug'). Proses dilewati.")

print("Jumlah gambar Cercospora sekarang:", len(os.listdir(source_folder)))

In [ ]:
# Reload dataset setelah penambahan gambar fisik teraugmentasi
train_data_final = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data_final = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Normalisasi
train_data_final = train_data_final.map(lambda x, y: (normalization_layer(x), y))
val_data_final = val_data_final.map(lambda x, y: (normalization_layer(x), y))

# Struktur Model Final
model_final = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:
model_final.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_final = model_final.fit(
    train_data_final,
    validation_data=val_data_final,
    epochs=20
)

# Simpan model final
model_final.save("model_paprika_final.h5")

In [ ]:
# Plot history
acc = history_final.history['accuracy']
val_acc = history_final.history['val_accuracy']
loss = history_final.history['loss']
val_loss = history_final.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='train')
plt.plot(epochs_range, val_acc, label='validation')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='train loss')
plt.plot(epochs_range, val_loss, label='validation loss')
plt.legend()
plt.title('Loss')
plt.show()

# Evaluasi Metrics
y_true = []
y_pred = []
for images, labels in val_data_final:
    preds = model_final.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=report_classes))

## 5. Pengujian Model Final

Bagian ini memuat model final yang telah disimpan dan mengujinya pada beberapa gambar tes (`tes 1.JPG`, `tes 2.jpg`, dan `tes 3.JPG`).

In [ ]:
# Load model final
model_final_loaded = load_model("model_paprika_final.h5")

# Jalankan pengujian pada gambar-gambar tes
test_images = ["samples/tes 1.JPG", "samples/tes 2.jpg", "samples/tes 3.JPG"]
for img_file in test_images:
    print(f"Menguji berkas: {img_file}")
    predict_and_plot(model_final_loaded, img_file, report_classes)